# Load the data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("combined_new.parquet")
print(df.shape)
df.head()


(7001613, 23)


,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,TAIL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,...,DEP_DEL15,ARR_DELAY,CANCELLED,CRS_ELAPSED_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,10,1,3,2025-10-01,AA,N102NN,JFK,"New York, NY",LAX,"Los Angeles, CA",...,0.0,-36.0,0.0,364.0,2475.0,NaN,NaN,NaN,NaN,NaN
1,10,1,3,2025-10-01,AA,N102NN,JFK,"New York, NY",SFO,"San Francisco, CA",...,1.0,18.0,0.0,389.0,2586.0,0.0,0.0,18.0,0.0,0.0
2,10,1,3,2025-10-01,AA,N102NN,LAX,"Los Angeles, CA",JFK,"New York, NY",...,1.0,76.0,0.0,333.0,2475.0,6.0,0.0,0.0,0.0,70.0
3,10,1,3,2025-10-01,AA,N102UW,DFW,"Dallas/Fort Worth, TX",BFL,"Bakersfield, CA",...,0.0,-23.0,0.0,210.0,1271.0,NaN,NaN,NaN,NaN,NaN
4,10,1,3,2025-10-01,AA,N102UW,DFW,"Dallas/Fort Worth, TX",IAH,"Houston, TX",...,0.0,0.0,0.0,75.0,224.0,NaN,NaN,NaN,NaN,NaN


In [2]:
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])
df = df.drop_duplicates()
print(df.shape)


(7001613, 23)


# Clean rows

Remove cancelled flights and rows where `DEP_DELAY` (our target source) is missing.

In [3]:
df = df[df["CANCELLED"] == 0].copy()
print("After dropping cancellations:", df.shape)


After dropping cancellations: (6898737, 23)


In [4]:
df = df.dropna(subset=["DEP_DELAY"])
print("After dropping missing target:", df.shape)


After dropping missing target: (6898737, 23)


# Create multiclass target

Bin `DEP_DELAY` into 5 classes: no delay, 15–29 min, 30–59 min, 1–2 hours, 2+ hours.

In [5]:
def categorize_delay(minutes):
    if minutes < 15:    return 0  # No delay
    elif minutes < 30:  return 1  # 15–29 min
    elif minutes < 60:  return 2  # 30–59 min
    elif minutes < 120: return 3  # 1–2 hours
    else:               return 4  # 2+ hours

df["DELAY_CLASS"] = df["DEP_DELAY"].apply(categorize_delay)
df["DELAY_CLASS"].value_counts(normalize=True).sort_index()

DELAY_CLASS
0    0.782692
1    0.071485
2    0.065401
3    0.047472
4    0.032951
Name: proportion, dtype: float64

# Feature engineering

Extract state from `"City, ST"`, and extract `DEP_HOUR` (0–23) from the HHMM `CRS_DEP_TIME`.


In [6]:
df["ORIGIN_STATE"] = df["ORIGIN_CITY_NAME"].str.split(",").str[-1].str.strip()
df["DEST_STATE"]   = df["DEST_CITY_NAME"].str.split(",").str[-1].str.strip()

df["DEP_HOUR"] = df["CRS_DEP_TIME"] // 100


#### Build PREV_FLIGHT_DELAY

For each flight, we look up the same aircraft's previous flight that day (via `TAIL_NUM`) and use that flight's arrival delay as a feature. If the plane landed late on its last leg, the next flight almost always departs late too — the plane just isn't at the gate yet.

This is the strongest pre-flight feature in the delay literature. It's not leakage: the previous flight landed before the current flight's scheduled departure.


In [7]:
# Sort chronologically within each aircraft
df = df.sort_values(["TAIL_NUM", "FL_DATE", "CRS_DEP_TIME"]).reset_index(drop=True)

# Shift ARR_DELAY down by one row within each TAIL_NUM group
df["PREV_ARR_DELAY"] = df.groupby("TAIL_NUM")["ARR_DELAY"].shift(1)
df["PREV_FL_DATE"]   = df.groupby("TAIL_NUM")["FL_DATE"].shift(1)

# Only use the previous leg if it was on the same day (legitimate same-day turnaround)
same_day = df["PREV_FL_DATE"] == df["FL_DATE"]
df["PREV_FLIGHT_DELAY"] = np.where(same_day, df["PREV_ARR_DELAY"], np.nan)
df["HAS_PREV_FLIGHT"]   = same_day.astype(int)

# Fill NaN with 0 — first flight of the day for this aircraft, assume on-time start
df["PREV_FLIGHT_DELAY"] = df["PREV_FLIGHT_DELAY"].fillna(0)

df = df.drop(columns=["PREV_ARR_DELAY", "PREV_FL_DATE"])

print(f"HAS_PREV_FLIGHT=1: {df['HAS_PREV_FLIGHT'].mean():.1%} of flights")
print("PREV_FLIGHT_DELAY vs DELAY_CLASS (mean by bucket):")
df["_prev_bkt"] = pd.cut(df["PREV_FLIGHT_DELAY"], bins=[-1e9, 0, 15, 60, 1e9],
                         labels=["on-time", "slight", "15-59", "60+"])
print(df.groupby("_prev_bkt", observed=False)["DELAY_CLASS"].mean().round(3))
df = df.drop(columns=["_prev_bkt"])


HAS_PREV_FLIGHT=1: 74.9% of flights
PREV_FLIGHT_DELAY vs DELAY_CLASS (mean by bucket):
_prev_bkt
on-time    0.248
slight     0.409
15-59      1.252
60+        2.635
Name: DELAY_CLASS, dtype: float64


# Drop columns we don't need

Drop columns only known after the flight, redundant/constant columns, and high-cardinality city columns (keeping state instead).

In [8]:
drop_cols = [
    # used to build PREV_FLIGHT_DELAY, then dropped (leakage)
    "ARR_DELAY",
    # redundant with DEP_DELAY (just the positive-clipped version)
    "DEP_DELAY_NEW",
    # already filtered
    "CANCELLED",
    # redundant after feature construction
    "FL_DATE",
    "CRS_DEP_TIME",
    "ORIGIN_CITY_NAME",
    "DEST_CITY_NAME",
    # too many unique values for one-hot; already used to build PREV_FLIGHT_DELAY
    "TAIL_NUM",
    # drop airport codes — ORIGIN_STATE already captures location, fewer categories
    "ORIGIN",
    "DEST",
    # DEP_DELAY, DEP_DEL15, CARRIER_DELAY are kept for EDA and
    # dropped inside model_analysis.ipynb before training.
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(df.shape)
df.head()


(6898737, 19)


,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,DEP_DELAY,DEP_DEL15,CRS_ELAPSED_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DELAY_CLASS,ORIGIN_STATE,DEST_STATE,DEP_HOUR,PREV_FLIGHT_DELAY,HAS_PREV_FLIGHT
0,1,1,3,G4,-5.0,0.0,199.0,1334.0,NaN,NaN,NaN,NaN,NaN,0,FL,NY,10,0.0,0
1,1,1,3,G4,-3.0,0.0,223.0,1334.0,NaN,NaN,NaN,NaN,NaN,0,NY,FL,14,-32.0,1
2,1,2,4,G4,-3.0,0.0,148.0,865.0,NaN,NaN,NaN,NaN,NaN,0,FL,KY,6,0.0,0
3,1,2,4,G4,-9.0,0.0,147.0,865.0,NaN,NaN,NaN,NaN,NaN,0,KY,FL,9,-21.0,1
4,1,2,4,G4,-4.0,0.0,120.0,643.0,NaN,NaN,NaN,NaN,NaN,0,FL,NC,12,-21.0,1


# Save preprocessed data

In [9]:
df.to_parquet('combined_preprocessed.parquet', index=False, compression='zstd')

In [10]:
df = pd.read_parquet('combined_preprocessed.parquet')
print(df.shape)
df.head()

(6898737, 19)


,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,DEP_DELAY,DEP_DEL15,CRS_ELAPSED_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DELAY_CLASS,ORIGIN_STATE,DEST_STATE,DEP_HOUR,PREV_FLIGHT_DELAY,HAS_PREV_FLIGHT
0,1,1,3,G4,-5.0,0.0,199.0,1334.0,NaN,NaN,NaN,NaN,NaN,0,FL,NY,10,0.0,0
1,1,1,3,G4,-3.0,0.0,223.0,1334.0,NaN,NaN,NaN,NaN,NaN,0,NY,FL,14,-32.0,1
2,1,2,4,G4,-3.0,0.0,148.0,865.0,NaN,NaN,NaN,NaN,NaN,0,FL,KY,6,0.0,0
3,1,2,4,G4,-9.0,0.0,147.0,865.0,NaN,NaN,NaN,NaN,NaN,0,KY,FL,9,-21.0,1
4,1,2,4,G4,-4.0,0.0,120.0,643.0,NaN,NaN,NaN,NaN,NaN,0,FL,NC,12,-21.0,1
